# LG CNS 채용 공고 하이브리드 DB 구축 — 정형 + JSONB + 벡터 + 전문검색

`10_하이브리드DB_실습.ipynb`의 설계(R1 컬럼 설계 → R2 인덱스 4종 → R3 `hybrid_search()`)를 **실제 채용 공고 데이터**(`CNS_20260911.json`)에 적용합니다.

**실습 노트북과 달라지는 점**

- 테이블은 실습용 `documents`를 건드리지 않도록 새 테이블 `job_postings`를 만듭니다.
- 합성 벡터 대신 **BGE-M3 실제 임베딩(1024차원)** 을 씁니다. 이제는 "의미가 비슷한 직무"가 실제로 검색되어야 하기 때문입니다.
- 검색 단위는 **공고 1건이 아니라 모집 분야(`recList`) 1건**입니다. 신입 공고 하나에 AI·Robotics·Consulting 등 9개 분야가 들어 있어서, 공고 단위로 임베딩하면 서로 다른 직무의 의미가 한 벡터에 뒤섞이기 때문입니다. (13개 공고 → 40개 행)
- 재실행 안전(멱등)은 `(job_notice_id, rec_sector_id)` UNIQUE 제약 + `INSERT ... ON CONFLICT DO UPDATE`(upsert)로 보장합니다. 새 날짜의 JSON을 다시 넣으면 같은 공고는 갱신되고 새 공고만 추가됩니다.

**데이터가 어디에 들어가는가**

| 원본 필드 | 저장 위치 | 이유 |
|---|---|---|
| 공고 ID, 회사, 직무명, 경력 구분, 채용 유형, 근무지, 접수 시작/마감 | 정형 컬럼 | 모든 공고에 항상 있고, 같음·범위 비교로 자주 필터링 |
| 경력 구분 목록, 전형 단계, 전공 조건, 지원 자격, 부서명 | `metadata` JSONB | 공고마다 구성이 다르고 목록 형태(전형 단계 4~6개 등) |
| 담당업무 + 자격요건 + 우대사항 + 전공 | `content` → `embedding`, `content_tsv` | 의미 검색(벡터)과 키워드 검색(전문검색)의 대상 |

## Step 0 — 연결 및 pgvector 확장 준비

In [10]:
import os, re, json
from datetime import datetime
from pathlib import Path

import numpy as np
import psycopg
from psycopg.types.json import Json, Jsonb
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from bs4 import BeautifulSoup   # 공고 본문이 HTML이라 평문으로 바꾸는 데 사용

load_dotenv()

# 적재할 JSON 파일 목록 — 다른 날짜 스냅샷이나 다른 계열사 파일도 여기에 추가하면 같은 방식으로 들어갑니다.
JSON_FILES = ["CNS_20260911.json"]

pw = os.environ.get("PGPASSWORD", "postgres")
conn = psycopg.connect(host="localhost", port=5432, dbname="course_db", user="postgres", password=pw)

conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
conn.commit()
register_vector(conn)   # 확장을 켠 "뒤"에 등록해야 VECTOR 타입을 찾을 수 있습니다.

print("연결 완료 ->", conn.info.dbname,
      "| pgvector:", conn.execute("SELECT extversion FROM pg_extension WHERE extname='vector'").fetchone()[0])

연결 완료 -> course_db | pgvector: 0.8.6


## Step 1 — 원본 JSON 파싱 (HTML → 평문, 공고 → 모집 분야 단위 레코드)

원본에서 본문 필드(`detailContext`, `requiredItem`, `preferredItem`, `majorCodeName`)는 인라인 스타일이 잔뜩 붙은 HTML입니다. 그대로 임베딩하면 `-webkit-text-fill-color` 같은 CSS 문자열이 의미를 오염시키므로 먼저 평문으로 정리합니다.

In [3]:
def html_to_text(html):
    """HTML을 사람이 읽는 평문으로 변환합니다. <li>는 '• '로 시작하는 줄이 됩니다."""
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for li in soup.find_all("li"):
        li.insert(0, "• ")
    text = soup.get_text("\n").replace("\xa0", " ")          # &nbsp; -> 일반 공백
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in text.splitlines()]
    lines = [ln for ln in lines if ln]
    # <li> 안에 <p>가 중첩되면 '•'만 한 줄로 떨어지므로 다음 줄과 합칩니다.
    merged = []
    for ln in lines:
        if merged and merged[-1] == "•":
            merged[-1] = "• " + ln
        else:
            merged.append(ln)
    return "\n".join(merged)


def parse_dt(s):
    """'2026.09.14 18:00' -> datetime. 값이 없으면 None."""
    return datetime.strptime(s.strip(), "%Y.%m.%d %H:%M") if s else None


def build_records(raw, source_name):
    """공고 리스트(raw)를 모집 분야 1건 = 레코드 1건으로 펼칩니다."""
    records = []
    for notice in raw:
        d = notice["jobNoticesDetail"]
        prc = notice.get("prcList") or []
        # 전형 단계: 공고마다 '서류 전형'/'서류전형'처럼 띄어쓰기가 달라서 공백을 없애 표준화합니다.
        #  -> JSONB 포함 검색(@>)은 문자열이 정확히 같아야 매치되기 때문입니다.
        steps = [p["displayName"].replace(" ", "") for p in prc if p.get("displayName")]
        step_schedule = {p["displayName"].replace(" ", ""): p["stepSchedule"]
                         for p in prc if p.get("displayName") and p.get("stepSchedule")}
        # '신입/경력'처럼 두 값이 한 문자열에 들어 있는 경우가 있어 목록으로 쪼갭니다.
        career_raw = (d.get("careerTypeName") or "").strip()
        career_types = [c.strip() for c in career_raw.split("/") if c.strip()]
        qualification = html_to_text(d.get("qualForAppInfo"))

        for r in notice.get("recList") or []:
            duty      = html_to_text(r.get("detailContext")) or html_to_text(r.get("mainTask"))
            required  = html_to_text(r.get("requiredItem"))
            preferred = html_to_text(r.get("preferredItem"))
            major     = html_to_text(r.get("majorCodeName"))
            job_name  = (r.get("jobGroupName") or "").strip()

            content = "\n\n".join(part for part in [
                f"직무: {job_name}",
                f"[담당업무]\n{duty}" if duty else "",
                f"[자격요건]\n{required}" if required else "",
                f"[우대사항]\n{preferred}" if preferred else "",
                f"[전공]\n{major}" if major else "",
            ] if part)

            metadata = {
                "career_types": career_types,
                "steps": steps,
                "step_schedule": step_schedule,
                "major": major,
                "major_free": ("전공 무관" in major) or ("전공무관" in major),
                "qualification": qualification,
                "org_name": r.get("orgName"),
                "source": source_name,
            }
            records.append({
                "job_notice_id": int(d["jobNoticeId"]),
                "rec_sector_id": int(r["recSectorId"]),
                "company":       d.get("companyName"),
                "notice_name":   d.get("jobNoticeName"),
                "job_name":      job_name,
                "job_group":     d.get("jobGroupSh"),
                "career_type":   career_raw,
                "recruit_type":  d.get("recruitTypeName"),
                "location":      (r.get("locationName") or d.get("workLocation") or "").strip() or None,
                "rec_start":     parse_dt(d.get("recStartDate")),
                "rec_end":       parse_dt(d.get("recEndDate")),
                "title":         f"{d.get('jobNoticeName')} - {job_name}",
                "content":       content,
                "metadata":      metadata,
            })
    return records


records = []
for path in JSON_FILES:
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    records += build_records(raw, Path(path).name)

print(f"공고 → 모집 분야 레코드: {len(records)}건")
print("\n[샘플]", records[0]["title"])
print(records[0]["content"][:500])
print("\n[metadata]", json.dumps(records[0]["metadata"], ensure_ascii=False)[:300])

공고 → 모집 분야 레코드: 40건

[샘플] 2026년 하반기 LG CNS 신입사원 채용 - AI (AX)
직무: AI (AX)

[담당업무]
• AI 기술을 고객의 실제 비즈니스에 적용하여 새로운 서비스를 만들고 업무 혁신을 실현합니다.
• 최신 AI·딥러닝 기술을 분석하고 검증하여 실제 서비스 환경에 적용 가능한 AI 모델을 개발하고 최적화합니다.
• LLM(초거대언어모델) 기반의 Agentic AI 서비스를 기획하고, RAG·Prompt Engineering·Function Calling·MCP(Model Context Protocol) 등을 활용해 AI Agent/Multi-Agent 애플리케이션을 개발합니다.
• LangChain, LangGraph 등 Agent Framework와 Vector DB를 활용하여 고객사 비즈니스 환경에 맞는 AI 모델을 설계하고 구축/운영합니다.
• 데이터, AI, 클라우드 기술을 접목시킨 언어 AI 서비스(AI 챗봇, AI Contact Center 등)를 제공하며, 시각 AI 영역에서는 생산·물류 공정의 AI Vision 서비스로 불량률 감소와

[metadata] {"career_types": ["신입"], "steps": ["서류전형", "필기전형", "1차면접", "2차면접", "건강검진"], "step_schedule": {"필기전형": "10월 초"}, "major": "전공 무관", "major_free": true, "qualification": "- 2027년 2월 졸업 예정자 또는 기졸업자 ('27년 1월부터 Full-time 근무 가능한 지원자)\n- 국내/해외 학사 이상 학위 보유자\n- IT 기본 역량 보유자는 누구나 지원 가능 (전공무관)\n- 남성의 경우 군필자 또는 


## Step 2 — 테이블 생성 (R1 대응: 정형 컬럼 + JSONB + VECTOR + tsvector를 한 스키마에)

In [4]:
conn.execute("""
CREATE TABLE IF NOT EXISTS job_postings (
    id             SERIAL PRIMARY KEY,
    -- 정형 컬럼: 모든 공고에 항상 있고 같음/범위 비교로 필터링하는 값
    job_notice_id  INTEGER NOT NULL,
    rec_sector_id  INTEGER NOT NULL,
    company        VARCHAR(100),
    notice_name    TEXT,
    job_name       VARCHAR(200),
    job_group      VARCHAR(50),
    career_type    VARCHAR(20),
    recruit_type   VARCHAR(20),
    location       VARCHAR(100),
    rec_start      TIMESTAMP,
    rec_end        TIMESTAMP,        -- TEXT가 아니라 TIMESTAMP여야 '마감 전' 범위 비교가 올바르게 동작
    -- 검색 대상 텍스트
    title          TEXT,
    content        TEXT,
    -- 반정형: 공고마다 구성이 다른 값
    metadata       JSONB,
    -- 벡터: BGE-M3 임베딩
    embedding      VECTOR(1024),
    -- 전문검색: title + content로 자동 계산
    content_tsv    tsvector GENERATED ALWAYS AS
                   (to_tsvector('simple', coalesce(title,'') || ' ' || coalesce(content,''))) STORED,
    -- 같은 공고의 같은 모집 분야는 한 행만 -> upsert의 기준
    UNIQUE (job_notice_id, rec_sector_id)
);
""")
conn.commit()
print("job_postings 준비 완료, 현재 건수:",
      conn.execute("SELECT count(*) FROM job_postings").fetchone()[0])

job_postings 준비 완료, 현재 건수: 0


## Step 3 — 임베딩 모델 준비 (BGE-M3, 1024차원)

8교시에서 쓴 BGE-M3를 `sentence-transformers`로 불러옵니다. `normalize_embeddings=True`로 길이 1인 벡터를 만들어 코사인 거리(`<=>`)와 맞춥니다. BGE-M3는 검색 질의에 별도 instruction 접두어가 필요 없어서 문서와 질의를 같은 함수로 임베딩합니다.

> 8교시에 `FlagEmbedding`의 `BGEM3FlagModel`을 썼다면 `embed()` 안만 `model.encode(texts)["dense_vecs"]`로 바꾸면 됩니다.

In [5]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"
model = SentenceTransformer(EMBED_MODEL)
DIM = model.get_embedding_dimension()
assert DIM == 1024, f"VECTOR(1024) 컬럼과 차원이 다릅니다: {DIM}"

def embed(texts):
    """문자열 리스트 -> (N, 1024) float32 배열, 각 행은 단위벡터."""
    return model.encode(texts, normalize_embeddings=True, batch_size=8,
                        show_progress_bar=len(texts) > 1).astype(np.float32)

print("임베딩 모델 준비 완료:", EMBED_MODEL, DIM)

c:\Users\Admin\Desktop\실습용\my_llm_service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 34158.98it/s]


임베딩 모델 준비 완료: BAAI/bge-m3 1024


## Step 4 — 적재 (임베딩 계산 + upsert)

In [6]:
vectors = embed([f"{r['title']}\n{r['content']}" for r in records])

COLS = ["job_notice_id", "rec_sector_id", "company", "notice_name", "job_name", "job_group",
        "career_type", "recruit_type", "location", "rec_start", "rec_end", "title", "content",
        "metadata", "embedding"]
# ON CONFLICT ... DO UPDATE: 같은 (공고, 모집 분야)가 이미 있으면 새 값으로 갱신 -> 재실행·새 스냅샷 적재에 안전
# EXCLUDED.컬럼 = "이번에 넣으려다 충돌한 새 값"을 가리키는 PostgreSQL 예약어
update_cols = [c for c in COLS if c not in ("job_notice_id", "rec_sector_id")]
UPSERT_SQL = f"""
INSERT INTO job_postings ({", ".join(COLS)})
VALUES ({", ".join(["%s"] * len(COLS))})
ON CONFLICT (job_notice_id, rec_sector_id) DO UPDATE SET
    {", ".join(f"{c} = EXCLUDED.{c}" for c in update_cols)};
"""

rows = []
for r, vec in zip(records, vectors):
    values = [r[c] for c in COLS[:-2]] + [Json(r["metadata"]), vec]
    rows.append(tuple(values))

with conn.cursor() as cur:
    cur.executemany(UPSERT_SQL, rows)
conn.commit()

total, no_emb = conn.execute(
    "SELECT count(*), count(*) FILTER (WHERE embedding IS NULL) FROM job_postings").fetchone()
print(f"적재 완료 -> 총 {total}건, 임베딩 누락 {no_emb}건 (0이어야 정상)")
assert no_emb == 0

# 단위벡터 확인: vector_norm은 pgvector가 제공하는 벡터 길이 함수
print("벡터 길이 min/max:", conn.execute(
    "SELECT round(min(vector_norm(embedding))::numeric, 4), round(max(vector_norm(embedding))::numeric, 4) FROM job_postings"
).fetchone())
for row in conn.execute(
        "SELECT career_type, count(*) FROM job_postings GROUP BY 1 ORDER BY 2 DESC").fetchall():
    print(" ", row)

Batches: 100%|██████████| 5/5 [00:21<00:00,  4.30s/it]

적재 완료 -> 총 40건, 임베딩 누락 0건 (0이어야 정상)
벡터 길이 min/max: (Decimal('1.0000'), Decimal('1.0000'))
  ('경력', 21)
  ('인턴', 9)
  ('신입', 9)
  ('신입/경력', 1)


## Step 5 — 인덱스 4종 (R2 대응)

| 인덱스 | 대상 | 쓰이는 조건 |
|---|---|---|
| B-tree | `rec_end` | `rec_end >= 기준일` (마감 전 공고만) |
| GIN `jsonb_path_ops` | `metadata` | `metadata @> '{"career_types": ["신입"]}'` 등 포함 검색 |
| HNSW `vector_cosine_ops` | `embedding` | `ORDER BY embedding <=> 질의벡터` |
| GIN | `content_tsv` | `content_tsv @@ plainto_tsquery(...)` |

경력 구분을 정형 컬럼 `career_type = '신입'`이 아니라 JSONB 목록으로 필터링하는 이유: 원본에 `'신입/경력'` 같은 값이 있어서, 같음 비교로는 "신입도 지원 가능한 공고"를 놓치기 때문입니다. 목록으로 쪼개 두고 `@>`로 포함 여부를 물으면 정확합니다.

In [7]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_job_postings_rec_end ON job_postings (rec_end);")
conn.execute("CREATE INDEX IF NOT EXISTS idx_job_postings_metadata_gin ON job_postings USING GIN (metadata jsonb_path_ops);")
conn.execute("CREATE INDEX IF NOT EXISTS idx_job_postings_embedding_hnsw ON job_postings USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64);")
conn.execute("CREATE INDEX IF NOT EXISTS idx_job_postings_tsv_gin ON job_postings USING GIN (content_tsv);")
conn.commit()
conn.execute("ANALYZE job_postings;")
conn.commit()

for r in conn.execute("SELECT indexname FROM pg_indexes WHERE tablename='job_postings' ORDER BY indexname;").fetchall():
    print(" -", r[0])

 - idx_job_postings_embedding_hnsw
 - idx_job_postings_metadata_gin
 - idx_job_postings_rec_end
 - idx_job_postings_tsv_gin
 - job_postings_job_notice_id_rec_sector_id_key
 - job_postings_pkey


## Step 6 — `hybrid_search()` (R3 + 도전 2 대응)

질의는 벡터 대신 **자연어 문장**을 받아 함수 안에서 임베딩합니다. 필터는 모두 선택 사항이며, 지정된 것만 AND로 결합됩니다. 실습 노트북과 같은 원칙으로 **조건절 구조만 f-string, 값은 전부 `%s` + params** 로 넘깁니다.

In [8]:
def hybrid_search(conn, query, career_type=None, step=None, major_free_only=False,
                  keyword=None, as_of=None, k=5):
    """
    query          : 자연어 질의 (의미 검색)
    career_type    : '신입' / '경력' / '인턴'          -> JSONB @>
    step           : 전형 단계 포함 여부, 예) '필기전형' -> JSONB @>
    major_free_only: True면 '전공 무관' 분야만         -> JSONB @>
    keyword        : 반드시 포함되어야 할 단어         -> tsvector @@
    as_of          : 이 시각 기준 마감 전 공고만       -> B-tree 범위
    """
    qvec = embed([query])[0]
    conditions, params = ["embedding IS NOT NULL"], []

    # JSONB 조건은 하나의 딕셔너리로 합쳐 @> 한 번으로 검사 (GIN 인덱스 한 번 조회)
    jsonb_filter = {}
    if career_type:
        jsonb_filter["career_types"] = [career_type]
    if step:
        jsonb_filter["steps"] = [step.replace(" ", "")]   # 적재 때와 같은 방식으로 표준화
    if major_free_only:
        jsonb_filter["major_free"] = True
    if jsonb_filter:
        conditions.append("metadata @> %s")
        params.append(Json(jsonb_filter))

    if keyword:
        conditions.append("content_tsv @@ plainto_tsquery('simple', %s)")
        params.append(keyword)
    if as_of:
        conditions.append("rec_end >= %s")
        params.append(as_of)

    sql = f"""
        SELECT id, job_name, notice_name, career_type, rec_end,
               metadata->>'major' AS major,
               embedding <=> %s::VECTOR(1024) AS distance
        FROM job_postings
        WHERE {" AND ".join(conditions)}
        ORDER BY distance
        LIMIT %s;
    """
    # %s 순서: SELECT의 질의벡터 -> WHERE의 params -> LIMIT
    cur = conn.execute(sql, (qvec, *params, k))
    cols = [c.name for c in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def show(results, title):
    print(f"\n=== {title} ({len(results)}건) ===")
    for r in results:
        end = r["rec_end"].strftime("%Y-%m-%d") if r["rec_end"] else "-"
        print(f"  {r['distance']:.3f} | {r['career_type']:6s} | 마감 {end} | {r['job_name']}")

In [9]:
Q_AI = "LLM과 RAG를 활용해 고객사에 AI 서비스를 만들어 적용하는 일"

show(hybrid_search(conn, Q_AI), "필터 없음")
show(hybrid_search(conn, Q_AI, career_type="신입"), "신입만 (JSONB)")
show(hybrid_search(conn, "클라우드 인프라를 설계하고 운영하는 일", career_type="경력", keyword="클라우드"),
     "경력 + 키워드 '클라우드' (JSONB + tsvector)")
show(hybrid_search(conn, "데이터를 분석해 고객의 비즈니스 문제를 해결하는 일", step="필기전형", major_free_only=True),
     "필기전형이 있고 전공 무관 (JSONB)")
show(hybrid_search(conn, Q_AI, as_of=datetime(2026, 9, 28)),
     "2026-09-28 기준 마감 전 (B-tree)")


=== 필터 없음 (5건) ===
  0.359 | 신입     | 마감 2026-09-14 | AI (AX)
  0.425 | 신입     | 마감 2026-09-14 | Consulting
  0.427 | 인턴     | 마감 2026-12-31 | AI
  0.450 | 경력     | 마감 2026-09-30 | Salesforce 솔루션 컨설팅 및 PM
  0.452 | 경력     | 마감 2026-09-30 | AI Architect


UndefinedFunction: operator does not exist: jsonb @> json
LINE 6:         WHERE embedding IS NOT NULL AND metadata @> $2
                                                         ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.

## Step 7 — 검증: 함수 결과 = 직접 SQL, 실행 계획 확인

In [ ]:
# 1) 매치 0건 조합은 예외 없이 빈 리스트여야 함
print("존재하지 않는 조합:", len(hybrid_search(conn, Q_AI, career_type="신입", as_of=datetime(2099, 1, 1))), "건")

# 2) 함수 결과와 직접 작성한 SQL 결과의 id 순서가 같아야 함
qvec = embed([Q_AI])[0]
direct = conn.execute("""
    SELECT id FROM job_postings
    WHERE embedding IS NOT NULL AND metadata @> %s
    ORDER BY embedding <=> %s::VECTOR(1024) LIMIT 5
""", (Json({"career_types": ["신입"]}), qvec)).fetchall()
via_func = hybrid_search(conn, Q_AI, career_type="신입", k=5)
assert [r[0] for r in direct] == [r["id"] for r in via_func]
print("직접 SQL과 hybrid_search 결과 일치")

# 3) 실행 계획 — 실습 노트북의 explain_query와 같은 방식
def explain_query(conn, sql, params):
    plan = conn.execute(f"EXPLAIN (ANALYZE, FORMAT JSON) {sql}", params).fetchone()[0][0]
    def walk(node, depth=0):
        line = "  " * depth + node.get("Node Type", "?")
        if "Index Name" in node:
            line += f" using {node['Index Name']}"
        print(line)
        for c in node.get("Plans", []):
            walk(c, depth + 1)
    walk(plan["Plan"])
    print(f"-> {plan['Execution Time']:.3f} ms")

explain_query(conn, """
    SELECT id FROM job_postings
    WHERE metadata @> %s AND content_tsv @@ plainto_tsquery('simple', %s)
    ORDER BY embedding <=> %s::VECTOR(1024) LIMIT 5
""", (Json({"career_types": ["경력"]}), "클라우드", qvec))

**실행 계획 읽는 법**: 지금은 40건뿐이라 옵티마이저가 인덱스 대신 `Seq Scan` + `Sort`를 고르는 것이 정상입니다(실습 노트북 실측과 같은 이유). 인덱스는 다른 계열사·다른 날짜 공고가 쌓여 행이 수천 건 이상이 될 때 효과를 냅니다.

**알아둘 한계**

- `'simple'` 토크나이저는 조사를 분리하지 못해서, 본문에 '클라우드를'로만 나온 분야는 키워드 '클라우드'에 매치되지 않습니다. 그래서 `keyword`는 꼭 필요한 경우에만 쓰고, 기본은 벡터 검색에 맡기는 편이 안전합니다.
- upsert는 추가·갱신만 합니다. 새 스냅샷에서 사라진(내려간) 공고는 테이블에 남으므로, 마감 여부는 `as_of` 필터로 거르거나 별도로 정리해야 합니다.